# 01 – Label & feature construction → `clean_train.csv`

This notebook builds the churn training table for VIB mobile-banking customers from the 6 raw tables in `../dataset/`.

The logic is ported from the SQL script `1st version/Code Clean Data (1).sql` to pandas, adding the steps missing from the SQL (Tenure, Avg_trans_no/month, population filters). The result is checked against the reference dataset `dataset/Clean Train Data.csv` at the end of the notebook.

**Steps:**
1. Load data
2. Define the `churn` label
3. Filter the study population
4. Build features from the Customer, Transaction, Activity, Deposit, Lending and Card tables
5. Join tables & export `clean_train.csv`
6. Validate against the reference dataset

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path('../dataset')
OUT_FILE = Path('clean_train.csv')
REF_DATE = pd.Timestamp('2019-12-31')   # end of the observation period (year 2019)

## 1. Load data

`CUSTOMER_NUMBER` is read as a string (it has leading zeros) and then converted to an integer to serve as the join key. The Activity table is large (~16M rows, 700MB), so only the required columns are loaded.

In [2]:
def read(name, **kw):
    df = pd.read_csv(DATA / name, dtype={'CUSTOMER_NUMBER': str}, low_memory=False, **kw)
    df['CUSTOMER_NUMBER'] = df['CUSTOMER_NUMBER'].astype(int)
    return df

customer = read('1.Data_Customer.csv')
trans    = read('2.Data_MyVIB_Transaction.csv')
activity = read('3.Data_MyVIB_Activity.csv', usecols=['ACTIVITY_DATE', 'CUSTOMER_NUMBER', 'ACTIVITY_NAME'])
deposit  = read('4.Data_Deposit.csv')
lending  = read('5.Data_Lending.csv')
card     = read('6.Data_Card.csv')

for n, d in [('customer', customer), ('trans', trans), ('activity', activity),
             ('deposit', deposit), ('lending', lending), ('card', card)]:
    print(f'{n:9s} rows={len(d):>10,}  customers={d.CUSTOMER_NUMBER.nunique():>8,}')

customer  rows=   290,223  customers= 290,223
trans     rows= 1,418,030  customers=  52,488
activity  rows=16,132,675  customers=  77,741
deposit   rows= 1,258,424  customers= 223,817
lending   rows=   576,431  customers= 102,014
card      rows=   871,589  customers= 150,459


## 2. Define the `churn` label

Definition (from table `[7.Data_set]` in the SQL):
- Sort each customer's activity rows by date and compute `Datediff` = number of days between two consecutive activity rows (`LAG` over `ACTIVITY_DATE`).
- `max_datediff` = the largest gap between two activities of the customer in 2019.
- `churn = 1` if `max_datediff > 30` days **or** `max_datediff` is null (the customer has exactly 1 activity row); otherwise `churn = 0`.

> Note: gaps are computed over **every row** of the Activity table (same-day rows are not merged), mirroring the behaviour of `LAG` in SQL. A customer with several rows all on the same day has `max_datediff = 0` → `churn = 0`.

In [3]:
activity['ACTIVITY_DATE'] = pd.to_datetime(activity['ACTIVITY_DATE'], format='%m/%d/%Y')
act = activity.sort_values(['CUSTOMER_NUMBER', 'ACTIVITY_DATE'], kind='stable')
act['Datediff'] = act.groupby('CUSTOMER_NUMBER')['ACTIVITY_DATE'].diff().dt.days

label = act.groupby('CUSTOMER_NUMBER').agg(max_datediff=('Datediff', 'max'))
label['churn'] = ((label['max_datediff'] > 30) | label['max_datediff'].isna()).astype(int)
# max_datediff is left empty (NaN) for customers with a single activity row, as in the SQL

print('Activity period:', act.ACTIVITY_DATE.min().date(), '->', act.ACTIVITY_DATE.max().date())
print('Customers with activity:', len(label))
label['churn'].value_counts()

Activity period: 2019-01-01 -> 2019-12-31
Customers with activity: 77741


churn
0    52409
1    25332
Name: count, dtype: int64

## 3. Filter the study population

The study population consists of customers who:
1. were active on the MyVIB app in 2019 (appear in the Activity table);
2. have a valid verification method: `SMS`, `HARD_TOKEN`, `SMART_OTP` (SQL: delete CIFs with any other `VERIFY_METHOD`);
3. registered for e-banking before 2019-12-01 (`IB_REGISTER_DATE <= 2019-11-30`, i.e. Tenure ≥ 1 month);
4. are aged 18 to 59;
5. hold at least 1 current account (`max(COUNT_CA_ACCT) >= 1` in the Deposit table).

In [4]:
cust = customer.copy()
cust['IB_REGISTER_DATE'] = pd.to_datetime(cust['IB_REGISTER_DATE'], errors='coerce')

# Age = 2019 - birth year (equivalent to DATEDIFF(year, DOB, '2019-12-31') in SQL).
# Some birth years were mistyped as 0xxx (e.g. 0996 -> 1996), so 1000 is added to them.
birth_year = pd.to_numeric(cust['DATE_OF_BIRTH'].astype(str).str[:4], errors='coerce')
birth_year = birth_year.where(birth_year >= 1900, birth_year + 1000)
cust['Age'] = 2019 - birth_year

max_ca = deposit.groupby('CUSTOMER_NUMBER')['COUNT_CA_ACCT'].max()

steps = [
    ('1. Active on MyVIB in 2019',          cust['CUSTOMER_NUMBER'].isin(label.index)),
    ('2. Valid VERIFY_METHOD',              cust['VERIFY_METHOD'].isin(['SMS', 'HARD_TOKEN', 'SMART_OTP'])),
    ('3. IB_REGISTER_DATE <= 2019-11-30',   cust['IB_REGISTER_DATE'] <= '2019-11-30'),
    ('4. Age 18-59',                        cust['Age'].between(18, 59)),
    ('5. Has a current account',            cust['CUSTOMER_NUMBER'].map(max_ca).fillna(0) >= 1),
]
mask = pd.Series(True, index=cust.index)
print(f'{"All customers":40s} {len(cust):>8,}')
for name, cond in steps:
    mask &= cond
    print(f'{name:40s} {mask.sum():>8,}')

pop = cust[mask].copy()

All customers                             290,223
1. Active on MyVIB in 2019                 77,741
2. Valid VERIFY_METHOD                     77,735
3. IB_REGISTER_DATE <= 2019-11-30          69,710
4. Age 18-59                               68,758
5. Has a current account                   66,763


## 4. Build features

| Feature | Source | Definition |
|---|---|---|
| client_sex, staff_vib, sms, Verify_method, EB_register_channel | Customer | as is |
| Age | Customer | 2019 − birth year |
| Tenure | Customer | (2019-12-31 − IB_REGISTER_DATE) / 30, in months |
| No.Activity_Name | Activity | number of distinct activity types |
| Type_Trans_LV2 | Transaction | number of distinct transaction types (TRANS_LV2) |
| Total_trans_no | Transaction | sum of TRANS_NO |
| Avg_trans_no/month | Transaction | Total_trans_no / Tenure |
| Avg / Max / Min_Trans_Amount | Transaction | mean / max / min of TRANS_AMOUNT |
| No_CurrentAccount, No_TermDeposite | Deposit | max monthly number of CA / TD accounts |
| Avg / Max / Min_…_Balance | Deposit | mean / max / min of the monthly average balance |
| No_Loan, Avg / Max / Min_Loan_Balance | Lending | max number of loans; mean / max / min outstanding balance |
| No_CC, No_DC | Card | max number of credit / debit cards |

Customers with no data in the Transaction / Lending / Card tables are filled with 0.

In [5]:
# --- Customer
pop['Tenure'] = ((REF_DATE - pop['IB_REGISTER_DATE']).dt.days / 30).round(2)

# --- Activity
f_act = activity.groupby('CUSTOMER_NUMBER').agg(**{'No.Activity_Name': ('ACTIVITY_NAME', 'nunique')})

# --- Transaction
f_trans = trans.groupby('CUSTOMER_NUMBER').agg(
    Type_Trans_LV2=('TRANS_LV2', 'nunique'),
    Total_trans_no=('TRANS_NO', 'sum'),
    Avg_Trans_Amount=('TRANS_AMOUNT', 'mean'),
    Max_trans_amount=('TRANS_AMOUNT', 'max'),
    Min_trans_amount=('TRANS_AMOUNT', 'min'),
)

# --- Deposit
f_dep = deposit.groupby('CUSTOMER_NUMBER').agg(
    No_CurrentAccount=('COUNT_CA_ACCT', 'max'),
    Avg_CurrentAccount_Balance=('AVG_CA_BALANCE', 'mean'),
    max_CurrentAccount_Balance=('AVG_CA_BALANCE', 'max'),
    Min_CurrentAccount_Balance=('AVG_CA_BALANCE', 'min'),
    No_TermDeposite=('COUNT_TD_ACCT', 'max'),
    Avg_TermDeposite_Balance=('AVG_TD_BALANCE', 'mean'),
    max_TermDeposite_Balance=('AVG_TD_BALANCE', 'max'),
    min_TermDeposite_Balance=('AVG_TD_BALANCE', 'min'),
)

# --- Lending
f_loan = lending.groupby('CUSTOMER_NUMBER').agg(
    No_Loan=('COUNT_OF_LOAN', 'max'),
    Avg_Loan_Balance=('AVG_LOAN_AMOUNT', 'mean'),
    max_Loan_Balance=('AVG_LOAN_AMOUNT', 'max'),
    min_Loan_Balance=('AVG_LOAN_AMOUNT', 'min'),
)

# --- Card
f_card = card.groupby('CUSTOMER_NUMBER').agg(
    No_CC=('COUNT_CREDITCARD', 'max'),
    No_DC=('COUNT_DEBITCARD', 'max'),
)

## 5. Join tables & export `clean_train.csv`

In [6]:
df = (pop.set_index('CUSTOMER_NUMBER')
         .join([f_act, f_trans, f_dep, f_loan, f_card, label]))

num_fill = list(f_trans.columns) + list(f_loan.columns) + list(f_card.columns)
df[num_fill] = df[num_fill].fillna(0)

# Avg_trans_no/month = Total_trans_no / Tenure (unrounded Tenure = days / 30), rounded half up
tenure_days = (REF_DATE - df['IB_REGISTER_DATE']).dt.days
df['Avg_trans_no/month'] = np.floor(df['Total_trans_no'] * 30 / tenure_days + 0.5)

df = df.reset_index().rename(columns={
    'CUSTOMER_NUMBER': 'customer_number', 'CLIENT_SEX': 'client_sex', 'STAFF_VIB': 'staff_vib',
    'SMS': 'sms', 'VERIFY_METHOD': 'Verify_method', 'EB_REGISTER_CHANNEL': 'EB_register_channel',
})
df['IB_REGISTER_DATE'] = df['IB_REGISTER_DATE'].dt.strftime('%-m/%-d/%Y')

COLUMNS = ['customer_number', 'client_sex', 'Age', 'staff_vib', 'sms', 'Verify_method', 'IB_REGISTER_DATE',
           'Tenure', 'EB_register_channel', 'No.Activity_Name', 'Type_Trans_LV2', 'Total_trans_no',
           'Avg_trans_no/month', 'Avg_Trans_Amount', 'Max_trans_amount', 'Min_trans_amount',
           'No_CurrentAccount', 'Avg_CurrentAccount_Balance', 'max_CurrentAccount_Balance',
           'Min_CurrentAccount_Balance', 'No_TermDeposite', 'Avg_TermDeposite_Balance',
           'max_TermDeposite_Balance', 'min_TermDeposite_Balance', 'No_Loan', 'Avg_Loan_Balance',
           'max_Loan_Balance', 'min_Loan_Balance', 'No_CC', 'No_DC', 'max_datediff', 'churn']
df = df[COLUMNS]

# Amounts are rounded to integers (half up); term-deposit (TD) balances are kept as computed
td_cols = ['Avg_TermDeposite_Balance', 'max_TermDeposite_Balance', 'min_TermDeposite_Balance']
amount_cols = [c for c in COLUMNS if ('Amount' in c or 'amount' in c or 'Balance' in c) and c not in td_cols]
int_cols = amount_cols + ['Age', 'No.Activity_Name', 'Type_Trans_LV2', 'Total_trans_no', 'Avg_trans_no/month',
                          'No_CurrentAccount', 'No_TermDeposite', 'No_Loan', 'No_CC', 'No_DC', 'churn']
df[amount_cols] = np.floor(df[amount_cols] + 0.5)
df[int_cols] = df[int_cols].astype('int64')

Rows follow the order of the reference dataset (if available) because `train_test_split(..., random_state=4)` in notebook 02 depends on row order — this keeps the train/test split reproducible.

In [7]:
REF_FILE = DATA / 'Clean Train Data.csv'
ref = pd.read_csv(REF_FILE) if REF_FILE.exists() else None

if ref is not None:
    df = df.set_index('customer_number').loc[ref['customer_number']].reset_index()
else:
    df = df.sort_values('customer_number').reset_index(drop=True)

df.to_csv(OUT_FILE, index=False)
print(f'Saved {OUT_FILE}: {df.shape}')
print(df['churn'].value_counts().rename(index={0: 'non-churner', 1: 'churner'}))
print(f"Churn rate: {df['churn'].mean():.2%}")
df.head()

Saved clean_train.csv: (66763, 32)
churn
non-churner    43003
churner        23760
Name: count, dtype: int64
Churn rate: 35.59%


,customer_number,client_sex,Age,staff_vib,sms,Verify_method,IB_REGISTER_DATE,Tenure,EB_register_channel,No.Activity_Name,...,max_TermDeposite_Balance,min_TermDeposite_Balance,No_Loan,Avg_Loan_Balance,max_Loan_Balance,min_Loan_Balance,No_CC,No_DC,max_datediff,churn
0,398189,M,23,N,Y,SMART_OTP,11/9/2019,1.73,BRANCH,10,...,0.000000e+00,0.0,0,0,0,0,0,1,6.0,0
1,580494,F,42,N,Y,SMS,3/18/2019,9.60,BRANCH,10,...,0.000000e+00,0.0,0,0,0,0,0,1,30.0,0
2,658868,M,25,N,Y,SMART_OTP,8/13/2019,4.67,BRANCH,13,...,0.000000e+00,0.0,0,0,0,0,0,1,3.0,0
3,253047,F,22,N,N,SMS,2/12/2019,10.73,BRANCH,10,...,1.270000e+09,0.0,0,0,0,0,0,0,16.0,0
4,171374,M,33,N,Y,SMART_OTP,4/17/2019,8.60,BRANCH,14,...,0.000000e+00,0.0,0,0,0,0,0,2,32.0,1


## 6. Validate against the reference dataset `dataset/Clean Train Data.csv`

Every column must match the reference dataset (tolerance ≤ 1 unit for rounding).

In [8]:
assert ref is not None, 'Reference dataset not found for validation'
assert set(df.customer_number) == set(ref.customer_number), 'Customer set differs from the reference dataset'
a = df.set_index('customer_number')
b = ref.set_index('customer_number').loc[a.index]
assert (a['churn'] == b['churn']).all(), 'Label differs from the reference dataset'

report = []
for c in COLUMNS[1:]:
    if pd.api.types.is_numeric_dtype(b[c]):
        diff = (a[c] - b[c]).abs()
        report.append((c, (diff == 0).mean(), diff.max()))
    else:
        report.append((c, (a[c].astype(str) == b[c].astype(str)).mean(), np.nan))
report = pd.DataFrame(report, columns=['column', 'exact_match_rate', 'max_abs_diff']).set_index('column')

num_ok = report['max_abs_diff'].fillna(0) <= 1
assert num_ok.all(), report[~num_ok]
print(f'OK: {len(df):,} customers, label and {len(report)} columns match the reference dataset')
report

OK: 66,763 customers, label and 31 columns match the reference dataset


,exact_match_rate,max_abs_diff
column,,
client_sex,1.000000,NaN
Age,1.000000,0.000
staff_vib,1.000000,NaN
sms,1.000000,NaN
Verify_method,1.000000,NaN
IB_REGISTER_DATE,1.000000,NaN
Tenure,1.000000,0.000
EB_register_channel,1.000000,NaN
No.Activity_Name,1.000000,0.000
